# Trích xuất Image Embeddings cho Đồ án Tốt nghiệp (DATN)

Notebook chạy trên Kaggle (GPU) để:
1. Tải ảnh sản phẩm từ `image_url`
2. Dùng mô hình đa phương thức **Jina CLIP v2** (`jinaai/jina-clip-v2`, vision encoder EVA-02) để trích xuất vector đặc trưng ảnh (1024 chiều)
3. Lưu kết quả ra `image_embeddings.npy` + file metadata để dùng cho các bước sau (index FAISS, gợi ý sản phẩm...)

## 1. Cài đặt thư viện

Kaggle có sẵn `transformers` nhưng bản cài sẵn có bug khi nạp Jina CLIP v2 (so sánh nhầm `str` với `int` lúc sort state_dict), nên mình ghim cứng về bản `5.3.0` đã test chạy ổn.

In [ ]:
!pip install -q --upgrade "transformers==5.3.0" polars pillow tqdm requests pyarrow einops timm


## 2. Import thư viện & kiểm tra GPU

In [ ]:
import os
import io
import requests
import numpy as np
import polars as pl
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

ImageFile.LOAD_TRUNCATED_IMAGES = True  # tránh lỗi khi đọc ảnh bị khuyết/hỏng một phần

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## 3. Cấu hình đường dẫn dữ liệu

Ưu tiên chạy trên Kaggle (`/kaggle/input/...`), có fallback sang `../data` để test nhanh ở local trước khi submit.

In [ ]:
INPUT_DIR = "/kaggle/input/datn-stream-subset"
OUTPUT_DIR = "/kaggle/working"
IMAGE_DOWNLOAD_DIR = "/kaggle/working/downloaded_images"

MODEL_NAME = "jinaai/jina-clip-v2"

os.makedirs(IMAGE_DOWNLOAD_DIR, exist_ok=True)

items_path = os.path.join(INPUT_DIR, "items.parquet")
if not os.path.exists(items_path):
    # fallback để test nhanh ở local
    items_path = "../data/items.parquet"
    INPUT_DIR = "../data"
    IMAGE_DOWNLOAD_DIR = "../data/downloaded_images"
    os.makedirs(IMAGE_DOWNLOAD_DIR, exist_ok=True)

print(f"Loading items from: {items_path}")


## 4. Đọc dữ liệu sản phẩm

In [ ]:
df_items = pl.read_parquet(items_path)
print(f"Tổng số sản phẩm: {len(df_items)}")
df_items.head(3)


## 5. Tải ảnh sản phẩm về máy

Việc tải ảnh phụ thuộc mạng (I/O-bound) nên dùng nhiều luồng (`ThreadPoolExecutor`) chạy song song thay vì tải tuần tự từng ảnh.

In [ ]:
def download_single_image(item_id, url, save_dir):
    if not url:
        return item_id, False, "Missing URL"

    file_path = os.path.join(save_dir, f"{item_id}.jpg")
    if os.path.exists(file_path):
        return item_id, True, "Already Exists"

    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        response = requests.get(url, timeout=10, headers=headers)
        if response.status_code == 200:
            img = Image.open(io.BytesIO(response.content))
            img.convert("RGB").save(file_path, "JPEG")
            return item_id, True, "Success"
        else:
            return item_id, False, f"HTTP {response.status_code}"
    except Exception as e:
        return item_id, False, str(e)


Chạy tải ảnh song song với 32 luồng.

In [ ]:
download_tasks = df_items.select(["item_id", "image_url"]).to_dicts()
print(f"Bắt đầu tải song song {len(download_tasks)} ảnh sản phẩm...")

success_count = 0
failed_count = 0
failed_log = {}

with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {
        executor.submit(download_single_image, task["item_id"], task["image_url"], IMAGE_DOWNLOAD_DIR): task
        for task in download_tasks
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Đang tải hình ảnh"):
        item_id, success, message = future.result()
        if success:
            success_count += 1
        else:
            failed_count += 1
            failed_log[item_id] = message

print(f"Hoàn thành tải ảnh! Thành công: {success_count}, Thất bại: {failed_count}")
if failed_count > 0:
    print("Ví dụ 5 lỗi tải ảnh đầu tiên:", list(failed_log.items())[:5])


## 6. Định nghĩa Dataset để nạp ảnh cho model

Kế thừa `torch.utils.data.Dataset`, trả về ảnh PIL kèm `item_id`. Ảnh tải lỗi/thiếu thì thay bằng ảnh đen để không làm crash cả batch.

In [ ]:
class ProductImageDataset(Dataset):
    def __init__(self, df, image_dir):
        self.df = df
        self.image_dir = image_dir
        self.item_ids = df["item_id"].to_list()

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, idx):
        item_id = self.item_ids[idx]
        file_path = os.path.join(self.image_dir, f"{item_id}.jpg")
        try:
            if os.path.exists(file_path):
                img = Image.open(file_path).convert("RGB")
            else:
                img = Image.new("RGB", (224, 224), color=0)  # ảnh thiếu/lỗi -> ảnh đen thay thế
        except Exception:
            img = Image.new("RGB", (224, 224), color=0)
        return item_id, img


## 7. Tải mô hình Jina CLIP v2

Bản `transformers` cài sẵn trên Kaggle có 3 lỗi tương thích với Jina CLIP v2, cần vá (monkeypatch) trước khi load model:

1. **`dot_natural_key` so sánh nhầm kiểu dữ liệu** khi sort tên tham số trong state_dict → gây `TypeError: '<' not supported between instances of 'str' and 'int'`.
2. **Thiết bị `meta`**: `transformers` khởi tạo model trên thiết bị "ảo" `meta` trước khi nạp trọng số để load nhanh hơn, nên buffer chưa được tính giá trị thật → phải ép về `cpu`.
3. **Buffer non-persistent bị ghi đè bằng vùng nhớ rác**: sau khi nạp xong, `transformers` vẫn ghi đè các buffer không nằm trong checkpoint (RoPE `freqs_cos/sin`, `inv_freq`...) → phải dựng 1 model tham chiếu rồi copy lại đúng giá trị.

Lỗi thứ 3 nguy hiểm nhất vì không ném exception nào cả — model vẫn chạy nhưng ra vector NaN (hoặc sai lệch âm thầm). Vì vậy cuối phần này luôn có bước "smoke test" để bắt lỗi ngay, trước khi tốn hàng giờ GPU chạy trên toàn bộ dataset.

In [ ]:
def patch_dot_natural_key():
    # Lỗi 1/3: sửa hàm sort dùng khi nạp state_dict, không cho so sánh int với str trực tiếp.
    try:
        import transformers.core_model_loading as _core_model_loading
    except ImportError:
        return False
    if not hasattr(_core_model_loading, "dot_natural_key"):
        return False

    def _safe_dot_natural_key(s):
        return [(0, int(p)) if p.isdigit() else (1, p) for p in s.split(".")]

    _core_model_loading.dot_natural_key = _safe_dot_natural_key
    return True


In [ ]:
import contextlib
import gc
import torch.utils._device as _torch_device_mod
from torch.utils._device import _device_constructors


@contextlib.contextmanager
def meta_init_safe_load():
    # Lỗi 2/3: ép mọi tensor tạo trong lúc khởi tạo model về "cpu" thay vì "meta",
    # để buffer được tính giá trị thật ngay từ đầu.
    orig_call = _torch_device_mod.DeviceContext.__torch_function__

    def patched_call(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        if self.device.type == "meta" and kwargs.get("device") is None and func in _device_constructors():
            kwargs = dict(kwargs)
            kwargs["device"] = "cpu"
            return func(*args, **kwargs)
        return orig_call(self, func, types, args, kwargs)

    _torch_device_mod.DeviceContext.__torch_function__ = patched_call
    try:
        yield
    finally:
        _torch_device_mod.DeviceContext.__torch_function__ = orig_call


In [ ]:
def _iter_non_persistent_buffers(module, prefix=""):
    for name, buf in module._buffers.items():
        if buf is not None and name in module._non_persistent_buffers_set:
            yield (f"{prefix}.{name}" if prefix else name), buf
    for child_name, child in module.named_children():
        child_prefix = f"{prefix}.{child_name}" if prefix else child_name
        yield from _iter_non_persistent_buffers(child, child_prefix)


def restore_non_persistent_buffers(model):
    # Lỗi 3/3 (quan trọng nhất, đã kiểm chứng thực nghiệm): sau from_pretrained,
    # transformers vẫn ghi đè MỌI buffer non-persistent bằng torch.empty_like()
    # (vùng nhớ rác), bất kể patch ở trên đã tính đúng hay chưa.
    # Cách vá: dựng 1 model tham chiếu bằng constructor thẳng (bỏ qua from_pretrained
    # nên không bị ghi đè), rồi copy buffer từ đó sang model thật.
    targets = list(_iter_non_persistent_buffers(model))
    if not targets:
        return []

    with meta_init_safe_load():
        ref_model = type(model)(model.config)
    ref_lookup = dict(_iter_non_persistent_buffers(ref_model))

    restored = []
    for name, buf in targets:
        ref_buf = ref_lookup.get(name)
        if ref_buf is not None and ref_buf.shape == buf.shape:
            buf.data.copy_(ref_buf.data.to(device=buf.device, dtype=buf.dtype))
            restored.append(name)

    del ref_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    missing = [n for n, _ in targets if n not in restored]
    if missing:
        raise RuntimeError(f"Không khôi phục được {len(missing)} buffer: {missing[:10]}")
    return restored


Áp dụng các patch, load model, và khôi phục buffer.

In [ ]:
_patched = patch_dot_natural_key()
print(f"patch_dot_natural_key applied: {_patched}")

print(f"Khởi tạo Jina CLIP Model: {MODEL_NAME}...")
with meta_init_safe_load():
    model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32)
model = model.to(device)
model.eval()
print("Tải mô hình thành công.")

_restored = restore_non_persistent_buffers(model)
print(f"Đã khôi phục {len(_restored)} buffer non-persistent (RoPE, ...).")


**Smoke test:** kiểm tra nhanh 1 ảnh mẫu để chắc chắn model không sinh NaN, trước khi chạy tốn thời gian trên toàn bộ dataset.

In [ ]:
_smoke_img = Image.new("RGB", (224, 224), color=(128, 128, 128))
with torch.no_grad():
    _smoke_emb = model.encode_image([_smoke_img], convert_to_numpy=True, show_progress_bar=False)
_smoke_emb = np.asarray(_smoke_emb, dtype=np.float32)
assert not np.isnan(_smoke_emb).any(), "Model sinh vector NaN ngay ở bước kiểm tra nhanh - dừng lại, không chạy full dataset!"
print(f"Smoke test OK - vector ảnh mẫu không chứa NaN (shape={_smoke_emb.shape}, dtype={_smoke_emb.dtype}).")


## 8. Trích xuất đặc trưng cho toàn bộ dataset

Chạy theo batch qua `DataLoader`. Sau mỗi batch kiểm tra NaN ngay để dừng sớm nếu có lỗi, thay vì chạy hết vài giờ GPU rồi mới phát hiện embedding hỏng.

In [ ]:
dataset = ProductImageDataset(df_items, IMAGE_DOWNLOAD_DIR)
dataloader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    collate_fn=lambda batch: ([x[0] for x in batch], [x[1] for x in batch]),
)

all_embeddings = []
item_ids_ordered = []

print("Bắt đầu trích xuất Image Embedding...")
with torch.no_grad():
    for batch_ids, batch_imgs in tqdm(dataloader, desc="Trích xuất đặc trưng ảnh"):
        features = model.encode_image(batch_imgs, convert_to_numpy=True, show_progress_bar=False)
        features = np.asarray(features, dtype=np.float32)  # ép float32 tường minh cho cosine similarity sau này

        # Kiểm tra NaN ngay trên từng batch, dừng sớm thay vì lưu hết rồi mới phát hiện lỗi
        nan_mask = np.isnan(features).any(axis=1)
        if nan_mask.any():
            bad_ids = [batch_ids[j] for j in np.where(nan_mask)[0]]
            raise RuntimeError(f"Phát hiện {int(nan_mask.sum())} vector NaN, item_id lỗi: {bad_ids[:10]}")

        all_embeddings.append(features)
        item_ids_ordered.extend(batch_ids)

image_embeddings = np.vstack(all_embeddings).astype(np.float32)
print(f"Hoàn tất! Kích thước ma trận embeddings hình ảnh: {image_embeddings.shape}, dtype: {image_embeddings.dtype}")


## 9. Lưu kết quả

In [ ]:
assert not np.isnan(image_embeddings).any(), "Phát hiện NaN trong embeddings trước khi lưu!"
assert not np.isinf(image_embeddings).any(), "Phát hiện Inf trong embeddings trước khi lưu!"
assert image_embeddings.dtype == np.float32, f"Kiểu dữ liệu không mong đợi: {image_embeddings.dtype}"

emb_output_path = os.path.join(OUTPUT_DIR, "image_embeddings.npy")
np.save(emb_output_path, image_embeddings)
print(f"Đã lưu ma trận vector nhúng ảnh tại: {emb_output_path}")

df_metadata = pl.DataFrame({
    "index": list(range(len(item_ids_ordered))),
    "item_id": item_ids_ordered,
})
meta_output_path = os.path.join(OUTPUT_DIR, "image_embedding_metadata.parquet")
df_metadata.write_parquet(meta_output_path)
print(f"Đã lưu metadata hình ảnh tại: {meta_output_path}")


## 10. Kiểm tra lại file đã lưu

In [ ]:
loaded_embeddings = np.load(emb_output_path)
print(f"Kiểm tra kích thước file tải lại: {loaded_embeddings.shape}, dtype: {loaded_embeddings.dtype}")
assert not np.isnan(loaded_embeddings).any(), "File đã lưu chứa NaN!"
assert np.allclose(image_embeddings, loaded_embeddings), "Dữ liệu lưu bị lỗi!"
print("Kiểm tra hoàn tất: không có NaN, dữ liệu khớp với bản gốc trong bộ nhớ.")
